# Gemma 4B LoRA Fine-Tuning (End-to-End)

This notebook demonstrates a complete fine-tuning workflow for `google/gemma-3-4b-it`.

## 1) Install dependencies

In [1]:
%pip install -q transformers datasets accelerate peft bitsandbytes torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "google/gemma-3-4b-it"
HF_TOKEN = os.getenv("HF_TOKEN")
DRY_RUN = True  # set to False for real fine-tuning

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({"device": "cuda" if torch.cuda.is_available() else "cpu", "dry_run": DRY_RUN})

{'device': 'cpu', 'dry_run': True}


## 2) Build instruction dataset

In [3]:
examples = [
    {"instruction": "Write a Python function that computes factorial.", "response": "def factorial(n):\n    return 1 if n <= 1 else n * factorial(n - 1)"},
    {"instruction": "Explain overfitting in one sentence.", "response": "Overfitting happens when a model memorizes training data patterns and performs worse on unseen examples."},
    {"instruction": "Give three tips for stable fine-tuning.", "response": "Use lower learning rates, keep a validation split, and monitor checkpoint metrics."},
    {"instruction": "What does LoRA do?", "response": "LoRA injects trainable low-rank adapters while leaving most base model weights frozen."},
    {"instruction": "Why do we need an eval set?", "response": "An eval set tracks generalization so we can detect overfitting early."},
    {"instruction": "Define gradient accumulation.", "response": "Gradient accumulation simulates larger batch sizes by combining gradients across multiple mini-batches."},
    {"instruction": "What is transfer learning?", "response": "Transfer learning adapts knowledge from a pretrained model to a new task."},
    {"instruction": "How can I avoid catastrophic forgetting?", "response": "Use a balanced dataset and conservative updates to preserve prior capabilities."},
]

def to_prompt(item):
    return f"<start_of_turn>user\n{item['instruction']}<end_of_turn>\n<start_of_turn>model\n{item['response']}<end_of_turn>"

texts = [to_prompt(item) for item in examples]
dataset = Dataset.from_dict({"text": texts})
split_dataset = dataset.train_test_split(test_size=0.25, seed=SEED)
split_dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 6
    })
    test: Dataset({
        features: ['text'],
        num_rows: 2
    })
})

## 3) Load Gemma tokenizer + model and apply LoRA

In [4]:
if DRY_RUN:
    print("Dry-run mode enabled: skipping Gemma model download and training.")
else:
    if not HF_TOKEN:
        raise ValueError("Set HF_TOKEN before running full fine-tuning.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

    tokenized = split_dataset.map(tokenize, batched=True, remove_columns=["text"])

    model_kwargs = {"token": HF_TOKEN}
    if torch.cuda.is_available():
        model_kwargs["device_map"] = "auto"
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
    if torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)

    model = get_peft_model(model, LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ))

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="outputs/gemma-3-4b-lora",
            overwrite_output_dir=True,
            num_train_epochs=1,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            learning_rate=2e-4,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            logging_steps=1,
            report_to="none",
            fp16=torch.cuda.is_available(),
            seed=SEED,
        ),
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )

    trainer.train()
    trainer.save_model("outputs/gemma-3-4b-lora/final")
    tokenizer.save_pretrained("outputs/gemma-3-4b-lora/final")
    print("Saved to outputs/gemma-3-4b-lora/final")

Dry-run mode enabled: skipping Gemma model download and training.
